In [ ]:
import os
import pandas as pd
from datetime import datetime 
import duckdb
import unicodedata
import sys
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

In [ ]:
# Add the 'src' directory to the path
sys.path.append(os.path.abspath("src"))
import central_perm_flow.pipelines.data_processing.nodes as nodes_dproc
import central_perm_flow.pipelines.calac_activos_baj_grad.nodes as nodes_abg

# Importación de Fuentes

In [ ]:
central_calaca_ext  = catalog.load('central_calendario_extendido')
central_calaca_uptoday  = catalog.load('central_calendario_extendido_uptoday')
central_estaca_sd = catalog.load('central_estaca_sd')

## Importación de parámetros

In [ ]:
parameters = catalog.load("parameters")

In [ ]:
# bajas_calac
bajas_calac = parameters['bajas_calac']
graduados_calac = parameters['graduados_calac']

## Bajas

In [ ]:
bajas_calendario_academico = nodes_abg.momento_baja(
    # 1. El DataFrame (Objeto en memoria)
    central_estaca_sd=central_estaca_sd, 
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    fallback_weeks=parameters['graduados_calac']['graduation_fallback_weeks'],
    
    # 2. Valores extraídos de diccionarios de configuración
    central_col_fechadef=bajas_calac['central_col_fechadef'],
    central_col_fechatemp=parameters['bajas_calac']['central_col_fechatemp'],
    
    # 3. Nombre del dataset según el catálogo
    central_calaca= central_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 4. Parámetros de transformación y cruce
    left_on=parameters['bajas_calac']['merge_left_on'],
    right_on=parameters['bajas_calac']['merge_right_on'],
    group_key=parameters['bajas_calac']['central_calaca_col_cohorte_inicial'],
    sort_cols=parameters['bajas_calac']['central_calaca_col_sort']
)

In [ ]:
parameters['bajas_calac']['central_calaca_col_cohorte_inicial']

In [ ]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_desercion = (
    bajas_calendario_academico
    .loc[:, ['cohorte_inicial','nivel_academico', 'fecha_ingreso', 'semana_acumulada', 'di']]
    .groupby(['cohorte_inicial','nivel_academico',  'fecha_ingreso', 'semana_acumulada'])
    .agg({'di': 'sum'})
    .reset_index()
    .sort_values(by='di', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_desercion.style.bar(
    subset=['di'], 
    color='#f8766d', # Un rojo suave tipo "soft red"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

In [ ]:
bajas_calendario_academico = catalog.load('central_bajas_calendario_academico')

## Graduados

In [ ]:
graduados_calendario_academico = nodes_abg.momento_grado(
    # 1. Referencias a datasets/DataFrames
    central_estaca=central_estaca_sd,
    central_calaca=central_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 2. Configuración de lógica de negocio (Duraciones y Grado Inmediato)
   
    col_gi=parameters['graduados_calac']['graduation_col_gi'],
    
    # 3. Parámetro de seguridad (Semanas por defecto si falla el cálculo)
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    fallback_weeks=parameters['graduados_calac']['graduation_fallback_weeks'],
    
    # 4. Llaves para el cruce de tablas (Joins)
    join_left=parameters['graduados_calac']['graduation_join_keys_left'],
    join_right=parameters['graduados_calac']['graduation_join_keys_right']
)

In [ ]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_graduacion = (
    graduados_calendario_academico
    .loc[:, ['cohorte_inicial','nivel_academico','nivel', 'fecha_ingreso', 'semana_acumulada', 'gi']]
    .groupby(['cohorte_inicial','nivel_academico','nivel',  'fecha_ingreso', 'semana_acumulada'])
    .agg({'gi': 'sum'})
    .reset_index()
    .sort_values(by='gi', ascending=False)
    .head(100)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_graduacion.style.bar(
    subset=['gi'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

## Activos

In [ ]:
activos_calendario_academico = nodes_abg.momento_activos(
   central_estaca=central_estaca_sd,
    central_calaca=central_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 2. Configuración de lógica de negocio (Duraciones y Grado Inmediato)
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    col_di='di',
    col_gi='gi',
    
    # 3. Parámetro de seguridad (Semanas por defecto si falla el cálculo)
    fallback_weeks=None,
    
    # 4. Llaves para el cruce de tablas (Joins)
    join_left='fecha_activo',
    join_right='fecha_inicio',
    group_key = 'fecha_ingreso'
) 

In [ ]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_eng = (
    activos_calendario_academico
    .loc[:, ['cohorte_inicial','nivel_academico','nivel','programa', 'fecha_ingreso', 'semana_acumulada','max_semana_teorica', 'engi']]
    .groupby(['cohorte_inicial','nivel_academico','nivel','programa',  'fecha_ingreso', 'semana_acumulada','max_semana_teorica'])
    .agg({'engi': 'sum'})
    .reset_index()
    .sort_values(by='engi', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_eng.style.bar(
    subset=['engi'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

In [ ]:
top_diez_activos = (
    activos_calendario_academico
    .groupby(['cohorte_inicial','fecha_ingreso','nivel_academico','nivel','programa',  'semana_acumulada','max_semana_teorica'])
    .agg(activos=('ai', 'sum')) # <-- Definimos: nombre = (columna, operacion)
    .reset_index()
    .sort_values(by='activos', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_activos.style.bar(
    subset=['activos'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

In [ ]:
def consolidar_universo_academico(
    df_bajas: pd.DataFrame,
    df_graduados: pd.DataFrame,
    df_activos: pd.DataFrame
) -> pd.DataFrame:
    """
    Concatena los tres estados académicos y normaliza indicadores.
    """
    # Unimos los tres dataframes
    universo = pd.concat([df_bajas, df_graduados, df_activos], ignore_index=True)
    
    # Normalizamos la columna engi: si es nula (viene de bajas/grados), es 0
    if 'engi' in universo.columns:
        universo['engi'] = universo['engi'].fillna(0).astype(int)
        
    return universo

In [ ]:
df_bajas = catalog.load('central_bajas_calendario_academico')
df_graduados = catalog.load('central_graduados_calendario_academico')
df_activos = catalog.load('central_activos_calendario')

In [ ]:
df_consolidado = catalog.load('central_estados_calac')

In [ ]:
df_consolidado['semana_acumulada']

In [ ]:
top_diez_activos = (
    df_consolidado
    .groupby(['cohorte_inicial','fecha_ingreso'])
    .agg(estudiantes=('identificacion', 'count')) # <-- Definimos: nombre = (columna, operacion)
    .reset_index()
    .sort_values(by='estudiantes', ascending=False)
  
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_activos.style.bar(
    subset=['estudiantes'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook


In [ ]:
top_diez_activos.sort_values(by= 'fecha_ingreso').head(20)

In [ ]:
df_consolidado = catalog.load('central_estados_calac')

In [ ]:
df_consolidado.groupby(['cohorte_inicial','fecha_ingreso']).agg({'ai':'sum',
                                                 'di':'sum',
                                                 'gi':'sum',
                                                 'engi':'sum',})

In [ ]:
df_consolidado.columns

In [ ]:
df_consolidado.loc[:,['identificacion', 'codigo_sis', 'nombres',
       'usuario_institucional', 'alianza', 'cohorte', 'cohorte_inicial',
       'cohorte_actual', 'nivel', 'nivel_academico', 'programa', 'estado',
       'bloque', 'descuentos', 'fecha_ingreso', 'fecha_de_registro',
       'fecha_de_baja_t', 'fecha_de_baja_d', 'fecha_baja',
       'fecha_de_reingreso', 'fecha_grado', 'fecha_activo', 'tipo_baja',
       'motivo_baja', 'submotivo_baja', 'comentarios', 'periodo_raw']]